# Notebook 07: Prepare Comparison Data for Experiments B & C

**Purpose:** Merge ground truth + pseudo labels and create training datasets for:
- **Experiment B:** 13k GT + 7k pseudo (~20k total)
- **Experiment C:** 4k GT + 7k pseudo (~11k total)

**Strategy:**
- Stratified sampling for 4k GT subset (maintain source distribution)
- Validate data quality (no duplicates, paths exist, labels valid)
- Generate annotation files for training notebooks 09 & 10

**Environment:** VSCode or local Python (not Colab - data prep only)

## Setup

In [1]:
import os
import json
import random
from pathlib import Path
from collections import Counter, defaultdict
from typing import List, Tuple

# Set random seed for reproducibility
SEED = 59
random.seed(SEED)

print(f"✓ Random seed set to {SEED}")

✓ Random seed set to 59


In [2]:
# Auto-detect environment and set base paths
is_colab = 'COLAB' in os.environ or os.path.exists('/content')

if is_colab:
    BASE_DIR = "/content/drive/MyDrive/Project/ocr_ai_agent_coding"
    IMAGE_ROOT = "/content/Dataset/data"  # Dataset unzipped to /content
    print("🌐 Running on Google Colab")
else:
    # VSCode/local: use relative paths from notebook directory
    BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
    IMAGE_ROOT = os.path.join(BASE_DIR, "Dataset/data")
    print("💻 Running locally (VSCode)")

# Data paths
DATA_DIR = os.path.join(BASE_DIR, "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
CRAWLED_DIR = os.path.join(DATA_DIR, "crawled")

# Input files
GT_TRAIN_PATH = os.path.join(PROCESSED_DIR, "train_line.txt")
PSEUDO_LABELS_PATH = os.path.join(CRAWLED_DIR, "pseudo_labels_final_clean.txt")

# Output files
TRAIN_4K_SUBSET_PATH = os.path.join(PROCESSED_DIR, "train_4k_subset.txt")
EXPERIMENT_B_PATH = os.path.join(PROCESSED_DIR, "experiment_B_train.txt")
EXPERIMENT_C_PATH = os.path.join(PROCESSED_DIR, "experiment_C_train.txt")
METADATA_PATH = os.path.join(PROCESSED_DIR, "comparison_data_metadata.json")

print(f"\n📁 Paths:")
print(f"  BASE_DIR: {BASE_DIR}")
print(f"  IMAGE_ROOT: {IMAGE_ROOT}")
print(f"  GT train: {GT_TRAIN_PATH}")
print(f"  Pseudo labels: {PSEUDO_LABELS_PATH}")

💻 Running locally (VSCode)

📁 Paths:
  BASE_DIR: /home/khang/Projects/OCR_project/ocr_ai_agent_coding
  IMAGE_ROOT: /home/khang/Projects/OCR_project/ocr_ai_agent_coding/Dataset/data
  GT train: /home/khang/Projects/OCR_project/ocr_ai_agent_coding/data/processed/train_line.txt
  Pseudo labels: /home/khang/Projects/OCR_project/ocr_ai_agent_coding/data/crawled/pseudo_labels_final_clean.txt


## 1. Load and Validate Data

In [3]:
def read_annotation_file(filepath: str) -> List[Tuple[str, str]]:
    """
    Read annotation file (tab-separated: image_path\tlabel).
    Returns list of (image_path, label) tuples.
    """
    entries = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            line = line.rstrip('\n')  # Preserve internal whitespace
            if not line:
                continue
            
            parts = line.split('\t')
            if len(parts) != 2:
                print(f"  ⚠️ Line {line_num}: Expected 2 parts, got {len(parts)}")
                continue
            
            image_path, label = parts
            entries.append((image_path.strip(), label))
    
    return entries


# Load ground truth training data
print("📖 Loading ground truth training data...")
gt_train = read_annotation_file(GT_TRAIN_PATH)
print(f"  ✓ Loaded {len(gt_train)} ground truth samples")

# Load pseudo-labeled data
print("\n📖 Loading pseudo-labeled data...")
pseudo_labels = read_annotation_file(PSEUDO_LABELS_PATH)
print(f"  ✓ Loaded {len(pseudo_labels)} pseudo-labeled samples")

📖 Loading ground truth training data...
  ✓ Loaded 13090 ground truth samples

📖 Loading pseudo-labeled data...
  ✓ Loaded 2503 pseudo-labeled samples


In [4]:
# Check for duplicates between GT and pseudo data
print("\n🔍 Checking for duplicates...")

gt_paths = set(path for path, _ in gt_train)
pseudo_paths = set(path for path, _ in pseudo_labels)

overlap = gt_paths & pseudo_paths

if overlap:
    print(f"  ⚠️ WARNING: {len(overlap)} images appear in both GT and pseudo data!")
    print(f"  Sample overlaps: {list(overlap)[:5]}")
else:
    print(f"  ✓ No duplicates found between GT and pseudo data")

print(f"\n📊 Data summary:")
print(f"  Ground truth: {len(gt_train)} samples")
print(f"  Pseudo labels: {len(pseudo_labels)} samples")
print(f"  Total available: {len(gt_train) + len(pseudo_labels)} samples")


🔍 Checking for duplicates...
  ✓ No duplicates found between GT and pseudo data

📊 Data summary:
  Ground truth: 13090 samples
  Pseudo labels: 2503 samples
  Total available: 15593 samples


In [12]:
# Validate image paths exist (sample check)
print("\n🔍 Validating image paths (sample check)...")

def validate_paths_sample(entries: List[Tuple[str, str]], sample_size: int = 100) -> Tuple[int, int]:
    """
    Check if image files exist for a sample of entries.
    Returns (existing_count, missing_count).
    """
    sample = random.sample(entries, min(sample_size, len(entries)))
    existing = 0
    missing = 0
    missing_paths = []
    
    for image_path, _ in sample:
        # Handle absolute paths (GT) and relative paths (pseudo)
        if os.path.isabs(image_path):
            full_path = image_path
        else:
            # Pseudo labels have relative paths from crawled dir
            if image_path.startswith('data/'):
              full_path = os.path.join(BASE_DIR, "Dataset", image_path)
            else:
                full_path = os.path.join(BASE_DIR, image_path)  # Already full paths in pseudo_labels.txt
        
        if os.path.exists(full_path):
            existing += 1
        else:
            missing += 1
            missing_paths.append(full_path)
    
    return existing, missing, missing_paths

gt_exist, gt_miss, gt_miss_paths = validate_paths_sample(gt_train, 100)
print(f"  GT sample (n=100): {gt_exist} exist, {gt_miss} missing")
if gt_miss > 0:
    print(f"    Missing paths: {gt_miss_paths[:3]}...")

pseudo_exist, pseudo_miss, pseudo_miss_paths = validate_paths_sample(pseudo_labels, 100)
print(f"  Pseudo sample (n=100): {pseudo_exist} exist, {pseudo_miss} missing")
if pseudo_miss > 0:
    print(f"    Missing paths: {pseudo_miss_paths[:3]}...")


🔍 Validating image paths (sample check)...
  GT sample (n=100): 100 exist, 0 missing
  Pseudo sample (n=100): 100 exist, 0 missing


## 2. Analyze Source Distribution

In [13]:
def extract_source_prefix(image_path: str) -> str:
    """
    Extract source dataset prefix from image filename.
    Examples:
      - UIT_HWDB_line_flat__40_4.jpg -> UIT_HWDB
      - VNOnDB_line__20160604_0199_25463_tg_0_3.png -> VNOnDB
      - vn_handwritten_images__0120_samples.png -> vn_handwritten_images
      - c4f1e9ec8f4e2ac8_line002.jpg -> crawled
    """
    filename = os.path.basename(image_path)
    
    if filename.startswith('UIT_HWDB'):
        return 'UIT_HWDB'
    elif filename.startswith('VNOnDB'):
        return 'VNOnDB'
    elif filename.startswith('vn_handwritten_images'):
        return 'vn_handwritten_images'
    else:
        return 'crawled'  # Pseudo-labeled data and other crawled images



# Analyze source distribution in ground truth
print("📊 Analyzing source distribution in ground truth data...")
gt_sources = [extract_source_prefix(path) for path, _ in gt_train]
gt_source_counts = Counter(gt_sources)

print(f"\n  Total GT samples: {len(gt_train)}")
for source, count in sorted(gt_source_counts.items()):
    pct = 100.0 * count / len(gt_train)
    print(f"    {source}: {count} ({pct:.1f}%)")

# Calculate target counts for 4k subset (proportional)
print(f"\n📐 Calculating stratified sampling targets for 4k subset...")
TARGET_SIZE = 4000
target_counts = {}

for source, count in gt_source_counts.items():
    proportion = count / len(gt_train)
    target = int(TARGET_SIZE * proportion)
    target_counts[source] = target
    print(f"  {source}: {count} -> {target} samples ({proportion*100:.1f}%)")

print(f"  Total target: {sum(target_counts.values())} samples")

📊 Analyzing source distribution in ground truth data...

  Total GT samples: 13090
    UIT_HWDB: 5783 (44.2%)
    VNOnDB: 5837 (44.6%)
    vn_handwritten_images: 1470 (11.2%)

📐 Calculating stratified sampling targets for 4k subset...
  UIT_HWDB: 5783 -> 1767 samples (44.2%)
  VNOnDB: 5837 -> 1783 samples (44.6%)
  vn_handwritten_images: 1470 -> 449 samples (11.2%)
  Total target: 3999 samples


## 3. Stratified Sampling for 4k GT Subset

In [14]:
# Group GT data by source
print("🗂️ Grouping ground truth data by source...")
gt_by_source = defaultdict(list)
for entry in gt_train:
    source = extract_source_prefix(entry[0])
    gt_by_source[source].append(entry)

print(f"  Grouped into {len(gt_by_source)} sources")

# Perform stratified sampling
print(f"\n🎲 Performing stratified sampling (seed={SEED})...")
gt_4k_subset = []

for source, target_count in target_counts.items():
    available = gt_by_source[source]
    sample_size = min(target_count, len(available))
    
    # Random sample without replacement
    sampled = random.sample(available, sample_size)
    gt_4k_subset.extend(sampled)
    
    print(f"  {source}: sampled {sample_size}/{len(available)} samples")

print(f"\n✓ Created 4k subset: {len(gt_4k_subset)} samples")

# Verify distribution
print(f"\n✓ Verifying 4k subset distribution...")
subset_sources = [extract_source_prefix(path) for path, _ in gt_4k_subset]
subset_source_counts = Counter(subset_sources)

for source, count in sorted(subset_source_counts.items()):
    pct = 100.0 * count / len(gt_4k_subset)
    original_pct = 100.0 * gt_source_counts[source] / len(gt_train)
    print(f"  {source}: {count} ({pct:.1f}%) [original: {original_pct:.1f}%]")

🗂️ Grouping ground truth data by source...
  Grouped into 3 sources

🎲 Performing stratified sampling (seed=59)...
  UIT_HWDB: sampled 1767/5783 samples
  VNOnDB: sampled 1783/5837 samples
  vn_handwritten_images: sampled 449/1470 samples

✓ Created 4k subset: 3999 samples

✓ Verifying 4k subset distribution...
  UIT_HWDB: 1767 (44.2%) [original: 44.2%]
  VNOnDB: 1783 (44.6%) [original: 44.6%]
  vn_handwritten_images: 449 (11.2%) [original: 11.2%]


## 4. Generate Experiment Annotation Files

In [20]:
WRITE_FILES = True  # Set to True to actually write files to disk

def write_annotation_file(entries: List[Tuple[str, str]], output_path: str, write_files: bool = None) -> None:
    """
    Write annotation file in VietOCR format (tab-separated).
    If write_files is None, uses global WRITE_FILES flag.
    """
    if write_files is None:
        write_files = WRITE_FILES

    if write_files:
        with open(output_path, 'w', encoding='utf-8') as f:
            for image_path, label in entries:
                f.write(f"{image_path}\t{label}\n")
        print(f"  ✓ Wrote {len(entries)} entries to {os.path.basename(output_path)}")
    else:
        print(f"  (dry-run) Would write {len(entries)} entries to {os.path.basename(output_path)}")


print("📝 Generating experiment annotation files...\n")

# 1. Save 4k GT subset
print("1️⃣ Saving 4k GT subset...")
write_annotation_file(gt_4k_subset, TRAIN_4K_SUBSET_PATH)

# 2. Experiment B: 18k GT + 7k pseudo
print("\n2️⃣ Creating Experiment B (18k GT + 2.5k pseudo)...")
# Use ALL ground truth (13,090 samples)
experiment_b_data = gt_train + pseudo_labels
write_annotation_file(experiment_b_data, EXPERIMENT_B_PATH)

# 3. Experiment C: 4k GT + 7k pseudo
print("\n3️⃣ Creating Experiment C (4k GT + 7k pseudo)...")
experiment_c_data = gt_4k_subset + pseudo_labels
write_annotation_file(experiment_c_data, EXPERIMENT_C_PATH)

print("\n✅ All annotation files generated successfully!")


📝 Generating experiment annotation files...

1️⃣ Saving 4k GT subset...
  ✓ Wrote 3999 entries to train_4k_subset.txt

2️⃣ Creating Experiment B (18k GT + 2.5k pseudo)...
  ✓ Wrote 15593 entries to experiment_B_train.txt

3️⃣ Creating Experiment C (4k GT + 7k pseudo)...
  ✓ Wrote 6502 entries to experiment_C_train.txt

✅ All annotation files generated successfully!


## 5. Generate Summary Statistics

In [17]:
# Analyze source distribution in each experiment
def analyze_experiment_sources(entries: List[Tuple[str, str]], name: str) -> dict:
    """
    Analyze source distribution in experiment data.
    """
    sources = [extract_source_prefix(path) for path, _ in entries]
    source_counts = Counter(sources)
    
    print(f"\n📊 {name}:")
    print(f"  Total samples: {len(entries)}")
    
    stats = {
        'total': len(entries),
        'sources': {}
    }
    
    for source, count in sorted(source_counts.items()):
        pct = 100.0 * count / len(entries)
        print(f"    {source}: {count} ({pct:.1f}%)")
        stats['sources'][source] = {
            'count': count,
            'percentage': round(pct, 2)
        }
    
    return stats

print("\n" + "="*60)
print("📈 EXPERIMENT DATA STATISTICS")
print("="*60)

baseline_stats = analyze_experiment_sources(gt_train, "Baseline (18k GT only)")
exp_b_stats = analyze_experiment_sources(experiment_b_data, "Experiment B (18k GT + 7k pseudo)")
exp_c_stats = analyze_experiment_sources(experiment_c_data, "Experiment C (4k GT + 7k pseudo)")


📈 EXPERIMENT DATA STATISTICS

📊 Baseline (18k GT only):
  Total samples: 13090
    UIT_HWDB: 5783 (44.2%)
    VNOnDB: 5837 (44.6%)
    vn_handwritten_images: 1470 (11.2%)

📊 Experiment B (18k GT + 7k pseudo):
  Total samples: 15593
    UIT_HWDB: 5783 (37.1%)
    VNOnDB: 5837 (37.4%)
    crawled: 2503 (16.1%)
    vn_handwritten_images: 1470 (9.4%)

📊 Experiment C (4k GT + 7k pseudo):
  Total samples: 6502
    UIT_HWDB: 1767 (27.2%)
    VNOnDB: 1783 (27.4%)
    crawled: 2503 (38.5%)
    vn_handwritten_images: 449 (6.9%)


## 6. Quality Checks

In [18]:
print("\n" + "="*60)
print("✅ QUALITY CHECKS")
print("="*60)

# 1. Check label lengths
print("\n1️⃣ Label length distribution...")

def check_label_lengths(entries: List[Tuple[str, str]], name: str, max_len: int = 180):
    lengths = [len(label) for _, label in entries]
    over_limit = sum(1 for l in lengths if l > max_len)
    
    print(f"\n  {name}:")
    print(f"    Mean length: {sum(lengths)/len(lengths):.1f} chars")
    print(f"    Max length: {max(lengths)} chars")
    print(f"    Over {max_len} chars: {over_limit} samples ({100*over_limit/len(entries):.1f}%)")
    
    if over_limit > 0:
        print(f"    ⚠️ WARNING: {over_limit} labels exceed max length {max_len}")

check_label_lengths(gt_train, "Ground truth")
check_label_lengths(pseudo_labels, "Pseudo labels")
check_label_lengths(experiment_b_data, "Experiment B")
check_label_lengths(experiment_c_data, "Experiment C")

# 2. Check for empty labels
print("\n2️⃣ Checking for empty labels...")

def check_empty_labels(entries: List[Tuple[str, str]], name: str):
    empty = sum(1 for _, label in entries if not label.strip())
    print(f"  {name}: {empty} empty labels")
    if empty > 0:
        print(f"    ⚠️ WARNING: Found {empty} empty labels")

check_empty_labels(gt_train, "Ground truth")
check_empty_labels(pseudo_labels, "Pseudo labels")
check_empty_labels(experiment_b_data, "Experiment B")
check_empty_labels(experiment_c_data, "Experiment C")

# 3. Print sample entries
print("\n3️⃣ Sample entries from each file...")

def print_samples(entries: List[Tuple[str, str]], name: str, n: int = 3):
    print(f"\n  {name} (first {n} entries):")
    for i, (path, label) in enumerate(entries[:n], 1):
        filename = os.path.basename(path)
        label_preview = label[:60] + '...' if len(label) > 60 else label
        print(f"    {i}. {filename}")
        print(f"       → \"{label_preview}\"")

print_samples(gt_4k_subset, "4k GT subset")
print_samples(experiment_b_data, "Experiment B")
print_samples(experiment_c_data, "Experiment C")

print("\n✅ Quality checks complete!")


✅ QUALITY CHECKS

1️⃣ Label length distribution...

  Ground truth:
    Mean length: 66.5 chars
    Max length: 158 chars
    Over 180 chars: 0 samples (0.0%)

  Pseudo labels:
    Mean length: 62.3 chars
    Max length: 126 chars
    Over 180 chars: 0 samples (0.0%)

  Experiment B:
    Mean length: 65.8 chars
    Max length: 158 chars
    Over 180 chars: 0 samples (0.0%)

  Experiment C:
    Mean length: 64.9 chars
    Max length: 158 chars
    Over 180 chars: 0 samples (0.0%)

2️⃣ Checking for empty labels...
  Ground truth: 0 empty labels
  Pseudo labels: 0 empty labels
  Experiment B: 0 empty labels
  Experiment C: 0 empty labels

3️⃣ Sample entries from each file...

  4k GT subset (first 3 entries):
    1. UIT_HWDB_line_flat__6_35.jpg
       → "tiếp tục công việc, công nhân đào vàng của Công ty Trường Sơ..."
    2. UIT_HWDB_line_flat__180_21.jpg
       → "rõ, chưa đầy đủ."
    3. UIT_HWDB_line_flat__254_3.jpg
       → "là một phần biển Đông."

  Experiment B (first 3 entries):


## 7. Save Metadata

In [19]:
# Save metadata about data preparation
metadata = {
    'created_date': '2026-03-23',
    'seed': SEED,
    'sampling_strategy': 'stratified',
    'source_files': {
        'ground_truth': os.path.basename(GT_TRAIN_PATH),
        'pseudo_labels': os.path.basename(PSEUDO_LABELS_PATH)
    },
    'output_files': {
        '4k_subset': os.path.basename(TRAIN_4K_SUBSET_PATH),
        'experiment_B': os.path.basename(EXPERIMENT_B_PATH),
        'experiment_C': os.path.basename(EXPERIMENT_C_PATH)
    },
    'baseline_stats': baseline_stats,
    'experiment_B_stats': exp_b_stats,
    'experiment_C_stats': exp_c_stats,
    'stratified_sampling': {
        'target_size': TARGET_SIZE,
        'target_counts': target_counts,
        'actual_counts': dict(subset_source_counts)
    }
}

with open(METADATA_PATH, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"\n💾 Metadata saved to {os.path.basename(METADATA_PATH)}")


💾 Metadata saved to comparison_data_metadata.json
